# Preprocessing & clustering

Standard preprocessing pipeline applied before downstream analyses.
Starting from a normalized `.h5ad` object, this notebook:

1. Regresses out technical variation (`total_counts`)
2. Scales gene expression
3. Runs PCA + neighbor graph + UMAP
4. Clusters with Leiden at a chosen resolution
5. Removes small clusters (< `MIN_CLUSTER_SIZE` cells) as a QC step

**Input:** `.h5ad` with `layers['transformed']` (log-normalized counts) and `obs['total_counts']`

**Output:** filtered `.h5ad` ready for cell type annotation or downstream analysis

## 1 · Imports

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2 · Settings

Edit this cell before running.

In [ ]:
INPUT_H5AD  = "/path/to/your/input.h5ad"
OUTPUT_H5AD = "/path/to/your/output_clustered.h5ad"

# ── QC filtering ─────────────────────────────────────────────────────────────
MIN_GENES    = 17    # minimum genes detected per cell
MIN_CELLS    = 1     # minimum cells expressing a gene

# ── PCA / neighbours ──────────────────────────────────────────────────────────
N_PCS        = 30    # PCs used for neighbour graph
N_NEIGHBORS  = 8     # k-NN graph size
UMAP_MIN_DIST = 0.001

# ── Clustering ───────────────────────────────────────────────────────────────
LEIDEN_RESOLUTIONS = [1.5]   # add more values to compare resolutions

# ── Small cluster filter ─────────────────────────────────────────────────────
MIN_CLUSTER_SIZE = 100   # clusters smaller than this are removed

## 3 · Load data

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Layers : {list(adata.layers.keys())}")
print(f"obs    : {list(adata.obs.columns)}")

## 4 · QC filtering & normalization

Filter low-quality cells and rarely-detected genes, compute QC metrics,
normalize to median library size, and log-transform.
All layers are stored for use in downstream tools.

In [ ]:
# ── QC filtering ─────────────────────────────────────────────────────────────
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"After QC filtering: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# ── QC metrics (before normalization, on raw counts) ─────────────────────────
sc.pp.calculate_qc_metrics(adata, inplace=True)

# ── Store raw counts ──────────────────────────────────────────────────────────
import scipy.sparse as sp
if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)
adata.layers["counts"] = adata.X.copy()

# ── Normalize to median library size ─────────────────────────────────────────
# target_sum=None uses the median library size across cells
sc.pp.normalize_total(adata, inplace=True)  # target_sum=1e4 for fixed 10k scaling
adata.layers["normalized"] = adata.X.copy()

# ── Log1p transform ───────────────────────────────────────────────────────────
sc.pp.log1p(adata)
adata.layers["transformed"] = adata.X.copy()

print(f"Max transformed value: {adata.layers['transformed'].max():.3f}")
print(f"Layers stored: {list(adata.layers.keys())}")

## 5 · Regression, scaling & PCA

Regress out library size effects, scale to unit variance, then run PCA.
Inspect the variance ratio to confirm `N_PCS` captures the main structure.

In [ ]:
# Set X to log-normalized values for scaling/PCA
adata.X = adata.layers["transformed"].copy()

# Regress out library size effect
sc.pp.regress_out(adata, ["total_counts"])

# Scale to unit variance
sc.pp.scale(adata)

# PCA
sc.tl.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

## 6 · Neighbour graph, UMAP & Leiden clustering

In [ ]:
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST)

for res in LEIDEN_RESOLUTIONS:
    key = f"leiden_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key)
    adata.obs[key] = (
        adata.obs[key]
        .astype("category")
        .cat.remove_unused_categories()
    )
    # reset auto-generated colours so they are recomputed on next plot
    if f"{key}_colors" in adata.uns:
        del adata.uns[f"{key}_colors"]
    n_clusters = adata.obs[key].nunique()
    print(f"Resolution {res} → {n_clusters} clusters")

sc.pl.umap(adata, color=[f"leiden_{res}" for res in LEIDEN_RESOLUTIONS], wspace=0.4)

## 7 · QC — identify small clusters

Clusters with fewer than `MIN_CLUSTER_SIZE` cells are likely to represent
doublets, debris, or poorly sampled populations. Inspect the size table
before deciding which clusters to remove.

In [ ]:
leiden_key = f"leiden_{LEIDEN_RESOLUTIONS[0]}"

cluster_sizes = adata.obs[leiden_key].value_counts().sort_index()
print(f"Cluster sizes ({leiden_key}):")
print(cluster_sizes.to_string())

small_clusters = cluster_sizes[cluster_sizes < MIN_CLUSTER_SIZE].index.tolist()
print(f"\nClusters below {MIN_CLUSTER_SIZE} cells: {small_clusters}")

## 8 · Remove small clusters

`clusters_to_remove` is auto-populated from the QC cell above.
Override manually if needed.

In [ ]:
# Override here if you want to keep or add specific clusters:
# clusters_to_remove = ["25", "26", "27"]
clusters_to_remove = small_clusters

n_before = adata.n_obs
adata = adata[~adata.obs[leiden_key].isin(clusters_to_remove)].copy()
n_after = adata.n_obs

print(f"Removed {len(clusters_to_remove)} cluster(s): {clusters_to_remove}")
print(f"Cells: {n_before:,} → {n_after:,} ({n_before - n_after:,} removed)")

# Refresh cluster colours
if f"{leiden_key}_colors" in adata.uns:
    del adata.uns[f"{leiden_key}_colors"]

sc.pl.umap(adata, color=[leiden_key], title=f"After filtering — {leiden_key}")

## 9 · Save

In [ ]:
# Restore log-normalized expression before saving
adata.X = adata.layers["transformed"].copy()

adata.write_h5ad(OUTPUT_H5AD)
print(f"Saved → {OUTPUT_H5AD}")
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  obs    : {list(adata.obs.columns)}")
print(f"  obsm   : {list(adata.obsm.keys())}")